# 08b. MAD-free residual 후보 모델 비교

08a 진단에서 유지된 R1 white noise, R2 OU, R3 signed Jump-OU를 validation condition에서 비교한다. R4 empirical replay는 비모수 reference이며 선택 대상이 아니다.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
OUTPUT = PROJECT_ROOT / 'outputs' / 'residual_model_selection_v2'

## 1. Validation 선택 및 high-rate test 실행

In [ ]:
completed = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'scripts' / 'run_residual_model_selection.py')],
    cwd=PROJECT_ROOT, check=True, capture_output=True, text=True
)
print(completed.stdout)

## 2. Validation 결과

점수는 분산, 분위수, ACF, ramp, PSD, OU innovation tail 및 innovation ACF 재현오차의 가중합이며 낮을수록 좋다.

In [ ]:
decision = json.loads((OUTPUT / 'selected_residual_model.json').read_text(encoding='utf-8'))
display(decision)
display(pd.read_csv(OUTPUT / 'validation_aggregate_metrics.csv').round(4))
display(Image(filename=str(OUTPUT / 'validation_model_scores.png')))

## 3. 81--100 request/s 외삽 test

Validation에서 선정된 family를 train+validation condition으로 다시 보정한 뒤 high-rate test에 적용한다.

In [ ]:
display(pd.read_csv(OUTPUT / 'test_aggregate_metrics.csv').round(4))
display(Image(filename=str(OUTPUT / 'representative_high_rate_test.png')))

## 4. 자동 생성 보고서

In [ ]:
display(Markdown((OUTPUT / 'residual_model_selection_report_ko.md').read_text(encoding='utf-8')))